# Step 3: SmoothQuant 量化（W8A8，两段式）+ 三方法对比 finale

**目标**：用 `llmcompressor` 的 **SmoothQuantModifier + GPTQModifier 两段式**把 Qwen2.5-7B-Instruct 量化成 **INT8 W8A8**（权重 + 激活都 8-bit）。SmoothQuant 通过数学等价的"平滑变换"把激活的离群点迁移到权重上，使**激活也能稳定地量化到 INT8**——这是它和 FP8（动态激活）/ AWQ（不量化激活）的根本区别。最后对比 FP8 / AWQ / SmoothQuant 三方法产物。

**对应 OUTLINE 课时**：2.5 SmoothQuant W8A8 全流程（~55 分钟）+ 2.6 三方法产物对比 finale。

> 为什么 SmoothQuant 要两段？第一段 `SmoothQuantModifier` 只是**改写模型权重**（插入平滑 scale，不量化）；第二段 `GPTQModifier(scheme="W8A8")` 才真把权重和激活压成 INT8。两段分开是因为"平滑"和"量化"是两个独立、可组合的步骤。

## 学完应能讲清（学完本节应能口头回答）

1. `oneshot` 通路三步？
2. SmoothQuant 为何是**两段** recipe（`SmoothQuantModifier` + `GPTQModifier(W8A8)`）？（先平滑：把激活离群点等价迁移到权重；后量化：纯 INT8 W8A8）
3. α 公式 `s_j=max(|X_j|)^α/max(|W_j|)^(1-α)` 与 `smoothing_strength` 是什么关系？（库默认 `smoothing_strength=0.5`=论文 α；课程取 0.8 偏向多迁权重）
4. 平滑为何是「等价变换」（前向不变）？
5. W8A8 产物怎么验证？

In [ ]:
%%capture
import subprocess, pathlib, json
import torch
import ipytest
ipytest.autoconfig()
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier
from llmcompressor.modifiers.transform.smoothquant import SmoothQuantModifier
from llmcompressor.modifiers.gptq import GPTQModifier

In [ ]:
# Setup cell（三 step 共用同一套模块根解析；规范见 course/NOTEBOOK_CONVENTIONS.md 第 2 节）。
# 每个 notebook 自包含地向上发现模块根（含 steps/ + pyproject.toml），绝不依赖裸相对路径或仓库根。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "steps").is_dir() and (cand / "pyproject.toml").exists():
            return cand
    raise RuntimeError("找不到模块根（含 steps/ + pyproject.toml 的目录）；请在模块目录内启动 jupyter")

MODULE_ROOT = _find_module_root(pathlib.Path.cwd())
MODEL_DIR      = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"      # 与 scripts/download_model.sh 一致
TINY_MODEL_DIR = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"     # L3 先在 0.5B 上验，再上 7B
OUT_ROOT       = MODULE_ROOT / "out"                                 # 已 gitignore
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MODULE_ROOT:", MODULE_ROOT)
print("GPU OK" if __import__("torch").cuda.is_available() else "无 GPU（仅 L1/L2 可跑）")


## 原理：SmoothQuant 为什么能让激活"可量化"

INT8 量化激活的难点是**离群点**（少数极大的激活通道，详见 M1）。直接 per-token INT8 会被离群点撑爆 scale、压死多数值。SmoothQuant 的洞察：

- 激活难量化（有离群点），但权重好量化（分布平滑）。
- 用一个数学等价变换把"难"从激活搬到权重：对每个 channel，给激活除 `s`、给对应权重列乘 `s`，前向输出不变，但激活离群点被压小、权重被放大一点（权重本来就好量化，无所谓）。
- 平滑强度 `smoothing_strength`（记作 α）控制搬多少：α 越大搬越多。论文取 α=0.5（权重与激活各占一半），llmcompressor 的 `SmoothQuantModifier` 类默认也正好是 α=0.5（与论文一致）。**本课程刻意取 α=0.8**，比论文更激进地把激活的离群点往权重搬——因为 Qwen2.5 的激活离群点较猛，经验上 0.8 平滑后 INT8 量化精度更稳。这个 0.8 由下面 `build_smoothquant_recipe` 的函数签名显式给出（学生从签名读到的就是 0.8）。

公式（per-channel）：`s_j = max(|X_j|)^α / max(|W_j|)^(1-α)`，然后 `X'_j = X_j / s_j`，`W'_j = W_j * s_j`。

`scheme="W8A8"` = 权重 per-channel 对称 INT8 + 激活 **dynamic per-token** 对称 INT8。**必须先 SmoothQuant 平滑**否则 INT8 激活量化精度崩。

**易错点（OUTLINE 标注）**：
- `SmoothQuantModifier` 在 `llmcompressor.modifiers.transform.smoothquant`（**不是** `modifiers.smoothquant`）。
- `ignore=["lm_head"]` 必须是**列表**（写字符串会报错）。


## 端到端：SmoothQuant 在通路上每步为什么这么干

**1. dataset 步——为何要校准？** 平滑要算每个 channel 的 `max(|X_j|)`（激活幅值）和 `max(|W_j|)`（权重幅值）定 scale → 需要校准数据前向收集激活。
**2. recipe 步——为何两段？** 第一段 `SmoothQuantModifier(smoothing_strength=...)`：算 per-channel scale `s`，做等价变换 `Y=(X/s)·(s·W)`，把激活离群点幅值迁进权重（激活变好量化、权重略难但 INT8 扛得住）。第二段 `GPTQModifier(scheme="W8A8")`：纯 INT8 量化已平滑的模型（GPTQ 用 Hessian 补偿权重，比纯 Round-To-Nearest 准）。
**3. oneshot 步**：前向收集激活统计 → 先平滑、后 GPTQ 量化 → 保存。
**4. 产物步**：读 `quantization_config`，确认 W8A8（权重+激活都 INT8）。

In [ ]:
## 亲手摸一摸：SmoothQuant 两段 modifier + 平滑 scale 长什么样？
from llmcompressor.modifiers.transform.smoothquant import SmoothQuantModifier
from llmcompressor.modifiers.gptq import GPTQModifier
import torch

sm = SmoothQuantModifier()              # 默认 smoothing_strength=0.5（=论文 α）
print("SmoothQuantModifier:", sm, "| 默认 smoothing_strength =", sm.smoothing_strength)
gptq = GPTQModifier(scheme="W8A8", targets="Linear", ignore=["lm_head"])
print("GPTQModifier:", gptq, "| .scheme =", gptq.scheme)

recipe = [sm, gptq]                      # 两段：先平滑、后量化
print("recipe:", recipe, "| 两段:", len(recipe))

# 平滑 scale 的直觉（合成激活/权重）
X = torch.randn(2, 8); W = torch.randn(8, 16)
alpha = 0.8
# W.abs().max(dim=1): 对每个输入通道 j，沿输出维取 max → 形状 (8,) 匹配 X 的 8 个通道
s = (X.abs().max(dim=0).values.pow(alpha) / W.abs().max(dim=1).values.pow(1-alpha)).clamp(min=1e-5)
print("scale s（每 channel 一个）:", s)
print("  -> 验证等价：X·W == (X/s)·(s*W)", torch.allclose(X@W, (X/s)@(s[:,None]*W), atol=1e-5))

## 本步填空

1. **`build_smoothquant_recipe(smoothing_strength, ignore)`** —— 构造两段式 recipe（SmoothQuantModifier + GPTQModifier），注意 `smoothing_strength` 默认 0.8。
2. **`smoothquant_config_summary(qc)`** —— 从 W8A8 产物抽出关键字段（INT8 权重 + INT8 动态激活，与 FP8 的 float 型对照）。
3. **`compare_methods(fp8_dir, awq_dir, sq_dir)`** —— finale：汇总三方法的位宽/显存/激活策略对比表（教学：理解三方法工程取舍）。

In [ ]:
def build_smoothquant_recipe(smoothing_strength=0.8, ignore=("lm_head",)):
    """返回 SmoothQuant 两段式 recipe（list of modifiers）。

    **为什么这么设计（填前先想）**：这个函数产出 s0 通路里的 `recipe`——一个 modifier 有序列表。SmoothQuant 要两段：第一段 SmoothQuantModifier 做等价变换把激活离群点幅值迁进权重（端到端第②步前半——只平滑、不量化），第二段 GPTQModifier(W8A8) 才真正把权重+激活压成 INT8（端到端第②步后半——GPTQ 用 Hessian 补偿比 RTN 准）。smoothing_strength 控制离群点幅值从激活搬多少到权重（α 越大搬越多）；课程取 0.8 覆盖库默认 0.5，因为 Qwen2.5 激活离群点更猛。

    返回 [第一段, 第二段] 两个 modifier 组成的 list：
      - 第一段：SmoothQuant 的"平滑变换"（只改写权重、不量化）。它需要一个控制
        平滑强度的参数——想想入参里哪个对应它。
      - 第二段：把权重 + 激活真正压成 INT8。用 GPTQModifier，它需要 targets（量化谁）、
        scheme（量化方案，本步固定为 "W8A8"）、ignore（跳过谁）三个参数。

    易错点（真坑，别踩）：
      - smoothing_strength 默认 0.8 由本函数签名显式给出（上面的默认参数）。
        注意：llmcompressor 的 SmoothQuantModifier 类默认是 0.5（与论文 α=0.5 一致），
        课程这里刻意取 0.8 以更激进地平滑 Qwen2.5 的激活离群点，所以把 0.8 写进函数签名
        来覆盖类的默认。调用方若不传 smoothing_strength，拿到的就是 0.8。
      - ignore 必须是 list；入参 ignore 是 tuple，需转换。
    """
    # TODO: 构造两个 modifier 并返回它们的 list。
    #   提示方向（不给具体实参）：
    #     1) 第一段 SmoothQuantModifier 传一个平滑强度参数（用入参 smoothing_strength）。
    #     2) 第二段 GPTQModifier 传 targets / scheme / ignore——scheme 用 "W8A8"；
    #        ignore 记得转成 list。
    raise NotImplementedError


# 脚手架（提供）：真正跑 SmoothQuant 的 execution
def run_smoothquant_quantize(model, tokenizer, calib_texts, save_dir, n_samples=512, seq_len=2048):
    recipe = build_smoothquant_recipe()
    oneshot(
        model=model, tokenizer=tokenizer,
        dataset=_build_calib_dataset(calib_texts, n_samples),
        recipe=recipe, max_seq_length=seq_len, num_calibration_samples=n_samples,
    )
    save_dir = pathlib.Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(save_dir, save_compressed=True)  # save_compressed 写 INT8 packed
    return save_dir


# 脚手架（提供）：校准数据包装（SmoothQuant 需要校准，INT8 W8A8 至少 512 样本）
def _build_calib_dataset(calib_texts, n_samples):
    from datasets import Dataset
    if calib_texts is not None:
        return Dataset.from_dict({"text": list(calib_texts)[:n_samples]})
    from datasets import load_dataset
    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    ds = ds.shuffle(seed=42).filter(lambda r: r["text"].strip())
    return ds.select(range(min(n_samples, len(ds))))

In [ ]:
def smoothquant_config_summary(quantization_config):
    """从 W8A8 (INT8) 产物的 quantization_config 抽出关键字段。

    **为什么这么设计（填前先想）**：这个函数读 s0 通路里的产物 `quantization_config`（端到端第④步），抽出证明「这是 INT8 W8A8」的关键字段。核心与非 INT8 方法的区分：`weights_type` 是 "int"（FP8 是 "float"）、`input_dynamic` 是 True（激活 dynamic per-token，AWQ 不量化激活故无此字段）。这些字段直接对应原理里的量化方案——权重 per-channel 对称 INT8、激活 dynamic per-token INT8。

    返回 dict，至少含：
      - "quant_method"        : str
      - "weights_num_bits"    : int   （8）
      - "weights_type"        : str   （"int"，与 FP8 的 "float" 对照）
      - "weights_symmetric"   : bool
      - "input_dynamic"       : bool  （W8A8 激活 dynamic per-token）
      - "input_num_bits"      : int   （8）
      - "targets"             : list
    """
    # TODO: 解析并返回
    raise NotImplementedError

In [ ]:
def compare_methods(fp8_dir, awq_dir, sq_dir):
    """汇总 FP8 / AWQ / SmoothQuant 三方法产物对比。

    **为什么这么设计（填前先想）**：这个函数是 finale——把三方法产物并排对比，让学员一眼看懂工程取舍。FP8 权重存 float8、SmoothQuant 权重存 int8——都是 8-bit 所以磁盘大小几乎相等（每个权重 1 字节），但类型不同（"float" vs "int"）。AWQ 权重存 int4——磁盘约是前两者的一半。激活策略差异更大：FP8 是 float8 dynamic、SmoothQuant 是 int8 dynamic（靠平滑才可能）、AWQ 不量化激活。这三行对比表回答了「到底用哪个」的工程问题。

    入参是三个产物目录路径（pathlib.Path 或 str，可能不存在——不存在则记 None）。
    返回 list of dict，每个 dict 一行：
      {"method": "FP8"/"AWQ"/"SmoothQuant",
       "weights_bits": int 或 None,
       "activations_bits": int 或 None（AWQ 不量化激活 -> None）,
       "disk_gb": float 或 None}
    依赖 smoothquant_config_summary（本步）与 s1/s2 的 summary 思路；这里直接读
    config_groups 的通用解析即可。
    """
    # TODO: 遍历三个目录，读 config.json -> quantization_config -> config_groups group_0，
    #       抽 weights.num_bits / input_activations.num_bits（无则 None），算 safetensors 总大小(GB)。
    raise NotImplementedError

In [ ]:
%%ipytest -qq

def test_build_smoothquant_recipe_two_stages():
    recipe = build_smoothquant_recipe()
    assert isinstance(recipe, list) and len(recipe) == 2
    assert isinstance(recipe[0], SmoothQuantModifier)
    assert isinstance(recipe[1], GPTQModifier)

def test_build_smoothquant_recipe_default_strength_is_08():
    recipe = build_smoothquant_recipe()
    # 函数签名显式给 smoothing_strength=0.8（类默认是论文的 0.5，课程刻意取 0.8）
    assert recipe[0].smoothing_strength == 0.8

def test_build_smoothquant_recipe_w8a8_and_ignore_list():
    recipe = build_smoothquant_recipe(ignore=("lm_head",))
    assert recipe[1].scheme == "W8A8"
    assert recipe[1].ignore == ["lm_head"]
    assert isinstance(recipe[1].ignore, list)  # OUTLINE：ignore 必须是 list

def test_smoothquant_config_summary_on_fake_w8a8():
    fake = {
        "quant_method": "compressed-tensors",
        "config_groups": {"group_0": {
            "targets": ["Linear"],
            "weights": {"num_bits": 8, "type": "int", "symmetric": True},
            "input_activations": {"num_bits": 8, "dynamic": True, "type": "int"},
        }},
    }
    s = smoothquant_config_summary(fake)
    assert s["quant_method"] == "compressed-tensors"
    assert s["weights_num_bits"] == 8
    assert s["weights_type"] == "int"
    assert s["weights_symmetric"] is True
    assert s["input_dynamic"] is True
    assert s["input_num_bits"] == 8
    assert s["targets"] == ["Linear"]

def test_compare_methods_handles_missing_and_present(tmp_path):
    # 构造两个假产物目录 + 一个不存在的
    import json as _json
    def make_fake(d, w_bits, a_bits, n_bytes):
        d.mkdir(parents=True, exist_ok=True)
        cg = {"group_0": {"targets": ["Linear"],
              "weights": {"num_bits": w_bits, "type": "int" if w_bits==8 else "float"}}}
        if a_bits is not None:
            cg["group_0"]["input_activations"] = {"num_bits": a_bits, "dynamic": True}
        (d / "config.json").write_text(_json.dumps({"quantization_config": {"config_groups": cg}}))
        # 小文件即可：compare_methods 只做「字节数累加 / 1e9」，不依赖真实体积，
        # 故不必写真 GB（写真 GB 会拖慢 L1、撑爆 /tmp）。
        (d / "model.safetensors").write_bytes(b"x" * n_bytes)
    fp8 = tmp_path / "fp8"; make_fake(fp8, 8, 8, n_bytes=2_000_000)
    awq = tmp_path / "awq"; make_fake(awq, 4, None, n_bytes=1_000_000)
    rows = compare_methods(fp8, awq, tmp_path / "missing")
    by = {r["method"]: r for r in rows}
    assert by["FP8"]["weights_bits"] == 8 and by["FP8"]["activations_bits"] == 8
    assert by["AWQ"]["weights_bits"] == 4 and by["AWQ"]["activations_bits"] is None
    # 存在的目录 disk_gb 应为正 float（验证「大小累加 + GB 换算」逻辑）
    assert isinstance(by["FP8"]["disk_gb"], float) and by["FP8"]["disk_gb"] > 0
    assert by["FP8"]["disk_gb"] > by["AWQ"]["disk_gb"]  # fp8 mock 文件比 awq 大
    assert by["SmoothQuant"]["disk_gb"] is None  # 目录不存在

## L2：tiny 模型验证（GPU 秒级 / CPU 可能分钟级）

用 tiny Qwen2 真跑 SmoothQuant 两段式 + GPTQ W8A8。验证：① recipe 结构对 ② SmoothQuant 平滑后 INT8 激活量化真能跑 ③ 产物是 INT8（type="int"）。

> **耗时提示**：GPTQ 第二段要对每个 Linear 做 Cholesky/Hessian 分解再逐列量化，比 FP8 的纯缩放慢。GPU 上 tiny 规模秒级；**纯 CPU 上可能十几秒到分钟级**——属正常，请耐心等待，不要中途中断。若 CPU 上随机权重的 Hessian 近奇异导致数值告警但不报错，产物仍可用于结构验证（本步目标是验证 recipe 跑通，不是精度）。

In [ ]:
from transformers import Qwen2Config, Qwen2ForCausalLM, PreTrainedTokenizerFast
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace

def make_tiny_tokenizer(vocab_size=320):
    vocab = {str(i): i for i in range(vocab_size)}
    tk = Tokenizer(WordLevel(vocab=vocab, unk_token="0"))
    tk.pre_tokenizer = Whitespace()
    return PreTrainedTokenizerFast(tokenizer_object=tk, unk_token="0", pad_token="0",
                                   eos_token="0", bos_token="0", model_max_length=128)

def make_tiny_model(vocab_size=320, hidden_size=64):
    cfg = Qwen2Config(num_hidden_layers=2, hidden_size=hidden_size,
                      intermediate_size=hidden_size * 2, num_attention_heads=2,
                      num_key_value_heads=2, vocab_size=vocab_size, tie_word_embeddings=True)
    return Qwen2ForCausalLM(cfg).eval()

tiny = make_tiny_model()
tiny_tok = make_tiny_tokenizer()
device = "cuda" if torch.cuda.is_available() else "cpu"
tiny.to(device)
print("tiny SmoothQuant 模型就绪:", sum(p.numel() for p in tiny.parameters()), "params")

calib_texts = [" ".join(str(i % 50) for i in range(60))] * 8

tiny_out = OUT_ROOT / "tiny-smoothquant"
run_smoothquant_quantize(tiny, tiny_tok, calib_texts, tiny_out, n_samples=8, seq_len=64)

qc = json.loads((tiny_out / "config.json").read_text())["quantization_config"]
summary = smoothquant_config_summary(qc)
print("SmoothQuant 产物摘要:", summary)
assert summary["weights_num_bits"] == 8
assert summary["weights_type"] == "int"
assert summary["input_num_bits"] == 8
assert summary["input_dynamic"] is True
print("L2 PASS：tiny SmoothQuant 两段式跑通，产物为 INT8 W8A8")

## L3：H200 执行（真 Qwen2.5-0.5B 再 7B）

GPU 守卫：无 GPU 自动跳过。SmoothQuant W8A8 需要 512 校准样本（INT8 激活量化对样本量敏感）。

> **⚠ 耗时提示（7B 必读，别误以为卡死）**：7B 的 SmoothQuant+GPTQ 两段式是本课程最慢的一步——
> GPTQ 第二段要对每个 Linear 逐层算 Hessian 再逐列量化，512 样本 × seq_len 2048 在 8×L20X/H200 上
> **实测约 15–25 分钟**才完成。对比之下 Step 1 (FP8) ~1 分钟、Step 2 (AWQ) ~5–8 分钟——
> 唯独本步 7B 有一个数量级的差距。
>
> 跑 7B cell 时：**进度条长时间不动是正常的**（GPTQ 逐层处理，每层耗时，进度更新粒度粗），
> 不要中途中断 kernel。若想确认没死，观察 GPU 利用率（`nvidia-smi`）应持续高位。
> 0.5B 那一段很快（约 1–2 分钟），慢的只是 7B 那一段。


In [ ]:
if torch.cuda.is_available():
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
    # 0.5B 与 7B 同属 Qwen2.5 族、tokenizer 词表一致（均为 Qwen2ForCausalLM），
    # 故这里复用 7B 的 tokenizer 给两者校准，无需再各下分一份。

    # 先 0.5B 快验
    m05b = AutoModelForCausalLM.from_pretrained(TINY_MODEL_DIR, torch_dtype="auto", device_map="auto")
    out_05b = OUT_ROOT / "qwen05b-smoothquant"
    run_smoothquant_quantize(m05b, tok, None, out_05b, n_samples=128, seq_len=512)
    print("0.5B SmoothQuant done ->", out_05b)
    del m05b; torch.cuda.empty_cache()

    # 再 7B（INT8 W8A8 需 512 样本）
    m7b = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype="auto", device_map="auto")
    out_7b = OUT_ROOT / "qwen7b-smoothquant"
    run_smoothquant_quantize(m7b, tok, None, out_7b, n_samples=512, seq_len=2048)
    print("7B SmoothQuant done ->", out_7b)
    del m7b; torch.cuda.empty_cache()
else:
    print("跳过：无 GPU（CPU 环境只跑 L1/L2）。")

## 产物检查 + Finale：三方法对比

打印 SmoothQuant 7B 产物，并用 `compare_methods` 汇总 FP8 / AWQ / SmoothQuant 三方法的位宽、激活策略、显存。

In [ ]:
# SmoothQuant 自身产物
sq_summary = None
sq_dir = OUT_ROOT / "qwen7b-smoothquant"
if sq_dir.exists():
    qc = json.loads((sq_dir / "config.json").read_text())["quantization_config"]
    sq_summary = smoothquant_config_summary(qc)
    print("== SmoothQuant 7B ==", sq_summary)
    sq_size = sum(f.stat().st_size for f in sq_dir.glob("*.safetensors"))
    print("   safetensors: {:.2f} GB".format(sq_size / 1e9))
else:
    print("(SmoothQuant 7B 产物不存在，可能 L3 未跑)")

# Finale：三方法对比（依赖 s1(qwen7b-fp8) / s2(qwen7b-awq) / 本步(qwen7b-smoothquant) 产物存在）
# 先体检三个产物目录，缺失的明确提示，避免把"没跑"误判成"代码错"。
finale_dirs = {
    "FP8":         OUT_ROOT / "qwen7b-fp8",          # 来自 Step 1 L3
    "AWQ":         OUT_ROOT / "qwen7b-awq",          # 来自 Step 2 L3
    "SmoothQuant": OUT_ROOT / "qwen7b-smoothquant",  # 来自本步 L3
}
missing = [name for name, d in finale_dirs.items() if not d.exists()]
print()
print("================ 三方法对比 finale ================")
if missing:
    print(f"⚠ 以下产物缺失: {missing} -> 对应行将显示 N/A。")
    print("  这不是你的代码错误，而是对应的 L3 执行 cell 未跑（或中途 OOM）。")
    print("  要看到完整三方法对比，请回 Step 1 / Step 2 / 本步 跑完各自的 L3。")
else:
    print("三个产物齐全，输出完整对比表。")
rows = compare_methods(finale_dirs["FP8"], finale_dirs["AWQ"], finale_dirs["SmoothQuant"])
header = "{:<14}{:<10}{:<12}{:<14}{}".format("方法", "权重位宽", "激活位宽", "磁盘(GB)", "激活策略")
print(header)
for r in rows:
    disk = "{:.2f}".format(r["disk_gb"]) if r["disk_gb"] else "N/A"
    act = "dynamic" if r["activations_bits"] else "不量化(A16)"
    print("{:<14}{:<10}{:<12}{:<14}{}".format(
        r["method"], str(r["weights_bits"]), str(r["activations_bits"]), disk, act))

print()
print("要点：FP8 = float8 权重+激活；AWQ = int4 权重、激活不量化（最省显存）；")
print("      SmoothQuant = int8 权重+激活（激活靠平滑才可量化）。")
print("      三者产物都是 compressed-tensors，vLLM 直接加载。")
print()
print("【反直觉，必读】为什么 FP8 与 SmoothQuant 的磁盘占用几乎相同（都≈FP16 的 50%+）？")
print("  磁盘上存的是*权重*，而两者都是 8-bit/权重：FP8 是 float8（1 个 E4M3 浮点/权重），")
print("  SmoothQuant 是 int8（1 个整数/权重）。8-bit 就是 8-bit，不管 int 还是 float，")
print("  每个权重都占 1 字节，所以磁盘几乎相等。它们的差别不在『占多少磁盘』，而在：")
print("    - FP8：激活也用 float8，H200 原生张量核加速、无需校准、精度最好；")
print("    - SmoothQuant：激活用 int8（靠平滑才稳），省的是『激活计算路径』而非权重磁盘。")
print("  而 AWQ 是 4-bit/权重，所以磁盘只有 ~25-30%（不到 FP8/SmoothQuant 的一半）。")
print("  即：『省显存』要分清省的是权重磁盘还是推理峰值显存——后者还含 KV cache 等，见 M4。")